# Ро и лямбда алгоритмы Полларда для получения дискретного логарифма
Мы узнали, как решить проблему дискретного логарифма в случае мультипликативной группы порядка $p-1$ с малыми делителями. Что же делать, если $p$ - безопасное простое число?  Для прямого перебора всех возможных решений DLP с $p$ порядка $40$ бит потребовалось бы несколько дней. Но есть способ получше.

Давайте повторим основу, нам даны: безопасное простое число $p$, генератор $g$, закрытый ключ $a$ и открытый ключ $A=g^a\ mod\ p$.

Ро-алгоритм Полларда - это вероятностный алгоритм, который позволяет решить проблему дискретного логарифма (DLP) за $O(\sqrt{p})$. Он назван $\rho$ из-за формы своего пространства поиска, как Вы можете увидеть на картинке, которую я бессовестно взял с википедии.

![pollardrhocycle.png](pollardrhocycle.png)

У алгоритма достаточно простая хоть и забавная основная идея. Мы создаём двух "кенгуру" (поэтому алгоритм ещё называется Кенгуру алгоритмом Полларда), одного Прирученного (Tame), одного Дикого (Wild), которые прыгают по пространству мультипликативной группы. Прыжки направляются детерминистической функцией $f$. К примеру вы можете использовать (при выполнении задания надо будет её немного поменять):

In [5]:
def f(x,m):
    return 1<<(x%m)

Где $m$ зависит от мощности группы $p-1$. По опыту хорошо работает $m=\frac{log_2(p-1)}{2}$

Один прыжок это:

$x_n=x_{n-1}+f(y_{n-1}), y_n=y_{n-1}\cdot g^{f(y_{n-1})}\ mod\ p$, так что $y_n=y_0*g^{x_n}$

Логика прыжков в том, что если оба кенгуру приземлятся на один элемент мультипликативной группы, они продолжат приземляться на одни и те же элементы, так как функция $f(x)$ - детерминистическая.

Вернёмся к нашим кенгуру.

Прирученный кенгуру стартует с конца:

$yT=g^d\ mod\ p, xT=d$
(заметьте, что мы не вычисляем $xT$ по модулю простого числа. Наша цель здесь получить итоговый логарифм **и** количество шагов, которое мы прошли)

и начинает прыгать ('^' обозначает возведение в степень):

```
for i in {0,k}
do
    xT = xT + f(yT)
    yT = yT * (g ^ f(yT)) % p
done
```

Можете попробовать разные $k$, хороший вариант - среднее всех возможных значений, генерируемых функцией $f$, помноженное на $4$.

Смысл в том, что Прирученный кенгуру пропрыгивает всю мультипликативную группу по кругу несколько раз и мы знаем дискретный логарифм финального элемента, на котором он останавливается.

Теперь начинаем прыгать Диким кенгуру:

```
xW = 0
yW = A
while xW <= xT
do
    xW = xW + f(yW)
    yW = yW * (g ^ f(yW)) % p
    if yW = yT
    then
        stop
    fi
done

```

Если $xT$ превышено, то надо поменять параметры и попробовать снова.

Алгоритм вероятностный и не гарантировано вычисляет логарифм при первой попытке. Однако, когда Вы перезапускаете алгоритм, надо поменять хеширующую функцию f(x), поскольку эта не сработала. Например, можно использовать 1<<((x\*k+l)%m)), где k и l выбираются случайно в начале каждой попытки использования алгоритма. Важно, чтобы k было не равно 0.

Но если найден $yW=yT$, то можно вычислить $a$:

$a=xT-xW\ mod\ p-1$

Идея заключается в том, что Дикий кенгуру пропрыгивает элементы мультипликативной группы, начиная с того, от которого мы хотим получить логарифм. Если в какой-то момент он попадет на один из тех элементов, на которых есть отпечатки лап Прирученный кенгуру, то он и дальше будет следовать за ним по пятам, не отклоняясь от маршрута Прирученный кенгуру. Мы следим за $xW$, чтобы знать, что сделано достаточно прыжков, чтобы догнать Прирученного. И если Дикий приземляется на нужный элемент в конце, то из разницы в прыжках мы можем легко восстановить логарифм стартового элемента.

Теперь попробуйте решить задачу локально с малыми параметрами:

In [156]:
import random

def pollard_lambda(A, g, p, max_tries=100):
    n = p - 1
    m = int(n.bit_length() / 2)

    for attempt in range(1, max_tries+1):
        print(f"Attempt: {attempt}")
        k = random.randrange(1, n)
        l = random.randrange(0, n)

        def step_func(y):
            return 1 << (((k * y + l) % n) % m)

        avg_step = (2**m - 1) / m
        kT = int(4 * avg_step)

        xT = n
        yT = pow(g, xT, p)
        for i in range(kT):
            s = step_func(yT)
            xT += s
            yT = (yT * pow(g, s, p)) % p
            if i % 10000 == 0:
                print(f"Jump: {i}")

        xW = 0
        yW = A
        max_steps = 2 * kT + 100000


        for i in range(max_steps):
            s = step_func(yW)
            xW += s
            yW = (yW * pow(g, s, p)) % p
            if yW == yT:
                a = (xT - xW) % n
                if pow(g, a, p) == A:
                    return a
            if i % 10000 == 0:
                print(f"i: {i}")

    return None

p = 882719
g = 2
a = random.randint(2, p-2)
A = pow(g, a, p)

print(f"Looking for a = {a}")
result = pollard_lambda(A, g, p)

if result == a:
    print("Success!")
else:
    print(f"Fail! Got {result}, expected {a}")

Looking for a = 700847
Attempt: 1
Jump: 0
i: 0
Success!


Рассмотрим другой случай. Что если $p$ 40 бит, но $a=k*b+c$, где длина $k$ - 20 бит, и  $k$ и $c$ известны?

В этой конфигурации проблему тоже достаточно просто решить, но надо использовать слегка отличающуюся версию алгоритма, которая называется ламбда-алгоритм Полларда (снова из-за того, как выглядит символ $\lambda$). Теперь цель не прыгать по всей группе, а гнаться друг за другом по прямой. Лямбда-версия работает, когда мы знаем, что логарифм будет в некотором интервале $(\mathit{start},\mathit{finish})$. В этом случае $m=\frac{log_2(\mathit{finish}-\mathit{start})}{2}$. 

Идея в том, что Прирученный кенгуру начинает с конца этого интервала и, используя примерно тот же алгоритм, что и при ро, но с немного отличающимися параметрами ($m$), прыгает не по всей группе, а по некоторой "взлетной полосе" за интервалом. Длина полосы определяется параметром $k$ и обычно длиннее интервала. После этого начинает прыгать Дикий кенгуру. Поскольку в начале он находится где-то в интервале, мы надеемся, что он приземлится на один из тех же элементов на полосе, что и Прирученный кенгуру и в итоге попадет на финальный элемент Прирученного.

Теперь вернемся к нашему случаю. Мы начинаем прыгать Прирученным с $yT=g^{k*l+c}$, где $l$ наибольшее целое, удовлетворяющее $k*l+c \lt p-1$. Мы используем $g^k$ в качестве генератора группы.


Далее для Дикого кенгуру также используем новый генератор, а $m$ заменяем на $\lceil\frac{p}{k}\rceil$. Тогда мы можем вычислить $b$ и соответственно $a$.

Попробуйте решить такую задачу локально:

In [212]:
from Crypto.Util.number import inverse as invert
import math

p=1088364193559
k=891239

b=random.randint(1,(p//k)-1)
a=b*k
g=2
A=pow(g,b*k,p)

def pollard_lambda_mult(A, g, p, k, L=None, U=None, dp_bits=14, tries=5):

    # group ord
    n = p - 1

    # interval bounds
    if L is None and U is None:
        L = 0
        U = (p - 2) // k
        interval_length = (p // k) - 1
    else:
        interval_length = U - L

    # new generator (c = 0)
    g_new = pow(g, k, p)
    A_norm = A % p

    m = math.floor(interval_length.bit_length() / 2)
    steps = [1 << i for i in range(m)]
    g_pow = [pow(g_new, s, p) for s in steps]
    
    avg_step = sum(steps) / len(steps)
    num_jumps = int(4 * (interval_length / avg_step))
    

    for attempt in range(tries):
        print(f"\nAttempt {attempt+1}/{tries}")
        
        salt1 = random.randint(1, 100)
        salt2 = random.randint(1, 100)
        
        def step_index(y):
            return (y + salt1 + salt2 * attempt) % m
        
        xT = U
        yT = pow(g_new, xT, p)
        dp_table = {}
        
        print(f"Kangaroo #1 starting from U={U}...")
        for i in range(num_jumps):
            j = step_index(yT)
            step = steps[j]
            xT += step
            yT = (yT * g_pow[j]) % p
            
            if (yT & ((1 << dp_bits) - 1)) == 0:
                dp_table[yT] = xT
            
            if (i + 1) % max(1, num_jumps//10) == 0:
                print(f"Kangaroo #1: {i+1}/{num_jumps} jumps, table size: {len(dp_table)}")
        
        print(f"Tame finished. Table size: {len(dp_table)}")
        
        xW = 0
        yW = A_norm
        print(f"Kangaroo #2 starting from A_norm...")
        
        for i in range(num_jumps * 2):
            j = step_index(yW)
            step = steps[j]
            xW += step
            yW = (yW * g_pow[j]) % p
            
            if (yW & ((1 << dp_bits) - 1)) == 0 and yW in dp_table:
                print(f"\nMeet on step {i}! xW={xW}, yW={yW}")
                xT_hit = dp_table[yW]
                b_candidate = (xT_hit - xW) % (U - L + 1)
                
                print(f"   xT_hit={xT_hit}, b_candidate={b_candidate}")
                
                if L <= b_candidate <= U:
                    a_candidate = (b_candidate * k) % n
                    if pow(g, a_candidate, p) == A:
                        print(f"Success: a={a_candidate}")
                        return a_candidate

            if (i + 1) % max(1, num_jumps//5) == 0:
                print(f"Kangaroo #2: {i+1}/{num_jumps*2} jumps, xW={xW}")
    
    print(f"\nFailed after {tries} attempts")
    return None

solution = pollard_lambda_mult(A, g, p, k)

if solution==a:
    print ("Success")
else:
    print ("Fail")


Attempt 1/5
Kangaroo #1 starting from U=1221181...
Kangaroo #1: 4774/47748 jumps, table size: 1
Kangaroo #1: 9548/47748 jumps, table size: 1
Kangaroo #1: 14322/47748 jumps, table size: 1
Kangaroo #1: 19096/47748 jumps, table size: 1
Kangaroo #1: 23870/47748 jumps, table size: 1
Kangaroo #1: 28644/47748 jumps, table size: 1
Kangaroo #1: 33418/47748 jumps, table size: 1
Kangaroo #1: 38192/47748 jumps, table size: 1
Kangaroo #1: 42966/47748 jumps, table size: 1
Kangaroo #1: 47740/47748 jumps, table size: 2
Tame finished. Table size: 2
Kangaroo #2 starting from A_norm...
Kangaroo #2: 9549/95496 jumps, xW=986010

Meet on step 10621! xW=1093089, yW=357066555392
   xT_hit=1503953, b_candidate=410864
Success: a=366178020496
Success


Теперь, когда вы протестировали функции локально, пора опробовать задание. Вам нужно решить две задачи и получить 2 флага.

In [213]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1346))
       
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения с сервера, по умолчанию до приглашенияк вводу"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попытайтесь снова подключиться к серверу.')
            return (None,None)
        if show:
            print (data)
        p=int(re.search(r'(?<=p=)\d+',data).group(0))
        A=int(re.search(r'(?<=2\*\*k \(mod p\)=)\d+',data).group(0))
        return (p,A)
    
    def checkSolution(self,k, show=True):
        """Проверка решения"""
        self.s.sendall((str(k)+'\n').encode())
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попытайтесь снова подключиться к серверу.')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        return False
    
    def checkSolution1(self,k,show=True):
        """Проверка первого решения"""
        return self.checkSolution(k,show)
    
    def checkSolution2(self,k,show=True):
        """Проверка второго решения"""
        return self.checkSolution(k,show)
    def __del__(self):
        self.s.close()



Задача 1

In [228]:
client = VulnServerClient()
g = 2
k = 1
print("Task #1")

L = 0
U = (1 << 32) - 1

p1, A1 = client.getChallenge()

solution = pollard_lambda_mult(A1, g, p1, k, L, U)

Task #1
Pollard lambda task stage 1
p=171665313024904820873835176917558615621908805927995361783049200071294720165140319009487773649839222149171536256594928626765343545825183619836629526151929875893011230541346064239190787012817219838093497715191476275073498533224910728429047207958735220059516675057047739716268899119570257410143231724776963564967
2**k (mod p)=60874708649990086012169294029457776193889071174125618267632177039543616914864025954372391065074614594914952423424141261565415615825613357053783160298515891673468126419546994188996573131411743069623526510089751843870843677912626866788967809541022682740261277565057292771352873869980338023566783985192785986630
0<k<2**32
Find k and send it to me:
>

Attempt 1/5
Kangaroo #1 starting from U=4294967295...
Kangaroo #1: 419436/4194368 jumps, table size: 283
Kangaroo #1: 838872/4194368 jumps, table size: 526
Kangaroo #1: 1258308/4194368 jumps, table size: 783
Kangaroo #1: 1677744/4194368 jumps, table size: 1017
Kangaroo #1: 2097180/4194368 j

In [215]:
client.checkSolution1(solution)

Congratulations, your flag is: DHTR{sm411_jump_b1g_jump}.
>


True

Далее, задание 2

In [229]:
client = VulnServerClient()

(p2,A2) = client.getChallenge()
print(p2, A2)

k1=10000
p1=832661519819

L = 0
U = (1 << 32) - 1

Pollard lambda task stage 1
p=171665313024904820873835176917558615621908805927995361783049200071294720165140319009487773649839222149171536256594928626765343545825183619836629526151929875893011230541346064239190787012817219838093497715191476275073498533224910728429047207958735220059516675057047739716268899119570257410143231724776963564967
2**k (mod p)=63120405063502671313593787756603417889894097265685883428569200532286115104428714519294455018888365530688772535009402753082130582742746170596833472058284671400106319718936259224708749667447995711143432350309944440728062960703091071922124877027969029289036321429768858331798047019845073974721514188698666296348
0<k<2**32
Find k and send it to me:
>
171665313024904820873835176917558615621908805927995361783049200071294720165140319009487773649839222149171536256594928626765343545825183619836629526151929875893011230541346064239190787012817219838093497715191476275073498533224910728429047207958735220059516675057047739716268899119570257410143231724776

In [231]:
solution2 = pollard_lambda_mult(A2, g, p2, k, L, U, dp_bits=14)


Attempt 1/5
Kangaroo #1 starting from U=4294967295...
Kangaroo #1: 419436/4194368 jumps, table size: 22
Kangaroo #1: 838872/4194368 jumps, table size: 44
Kangaroo #1: 1258308/4194368 jumps, table size: 64
Kangaroo #1: 1677744/4194368 jumps, table size: 82
Kangaroo #1: 2097180/4194368 jumps, table size: 104
Kangaroo #1: 2516616/4194368 jumps, table size: 122
Kangaroo #1: 2936052/4194368 jumps, table size: 143
Kangaroo #1: 3355488/4194368 jumps, table size: 157
Kangaroo #1: 3774924/4194368 jumps, table size: 173
Kangaroo #1: 4194360/4194368 jumps, table size: 192
Tame finished. Table size: 192
Kangaroo #2 starting from A_norm...

Meet on step 754633! xW=2798736265, yW=38357799742425422828794446539638544394695877855575150758454404953815118256784788534962778194500315012050374017054619645969941764908041504746578272441510747105187546793118274188005754413981169433410916603962711155693609629259658254763257859020591039750492056812935126847746811823508845890538170590390592143360
   xT_hit=46019

In [232]:
print(solution2)

1803235792


In [233]:
client.checkSolution2(solution2)

Congratulations, your flag is: DHTR{sm411_jump_b1g_jump}.
>


True